Created by: Varuna

Date: Oct 2025

# Imports

In [1]:
import pandas as pd
import json
import numpy as np

# directory path

In [2]:
directory_path = '/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd'

# Main data

In [3]:
df = pd.read_csv(f'{directory_path}/test.csv')

In [5]:
df.shape

(292, 129)

# Load GAMLSS data

In [6]:
gamlss_data = pd.read_csv('/projectnb/vkolagrp/projects/adrd_foundation_model/image_proc/gamlss_data/gamlss_thresholded_wtext_wmetadata_0805.csv')
nifd_gamlss = gamlss_data[gamlss_data['COHORT'] == 'NIFD']
print(nifd_gamlss.shape)
nifd_gamlss.head()

(231, 6)


,ID,COHORT,NACCAGE,SEX,DX,brain_findings_summary
2468,NIFD_1_S_0019,NIFD,71.0,1.0,CN,No significant findings. Brain MRI appears age...
2469,NIFD_1_S_0023,NIFD,70.0,1.0,CN,No significant findings. Brain MRI appears age...
2470,NIFD_1_S_0024,NIFD,71.0,2.0,CN,Inferior Ventricles region has moderate enlarg...
2471,NIFD_1_S_0034,NIFD,70.0,2.0,CN,Frontal region has mild atrophy; Inferior Vent...
2472,NIFD_1_S_0044,NIFD,84.0,1.0,CN,Occipital region has moderate atrophy


In [7]:
nifd_gamlss['LONI_ID'] = nifd_gamlss['ID'].str.split('NIFD_').str[1]

/scratch/ipykernel_3566279/3179193391.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nifd_gamlss['LONI_ID'] = nifd_gamlss['ID'].str.split('NIFD_').str[1]


# Merge GAMLSS and NIFD clinical

In [8]:
merged_df = pd.merge(df, nifd_gamlss[['LONI_ID', 'brain_findings_summary']], on = ['LONI_ID'], how='left')

In [9]:
merged_df.head()

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,DOB_convert,CLINICAL_LINKDATE_convert,AGE,COHORT,patient_summary,diag_summary,brain_findings_summary
0,1_S_0001,8/27/14,CON,UCSF,7,12/9/45,1,15.0,1,NaN,...,NaN,NaN,NaN,1945-12-09,2014-08-27,68.714579,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Healthy Control""}",NaN
1,1_S_0002,1/24/12,BV,UCSF,3,12/28/54,2,13.0,1,1/24/12,...,NaN,NaN,NaN,1954-12-28,2012-01-24,57.073238,NIFD,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}",NaN
2,1_S_0003,8/20/12,SV,UCSF,4,6/8/44,1,12.0,1,8/15/12,...,8/15/12,12.0,14.0,1944-06-08,2012-08-20,68.199863,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Semantic Variant FTD""}",Hippocampus region has severe atrophy; Medial ...
3,1_S_0005,8/26/14,BV,UCSF,6,11/22/51,1,18.0,1,8/26/14,...,8/26/14,-5.0,15.0,1951-11-22,2014-08-26,62.759754,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",NaN
4,1_S_0006,7/15/11,BV,UCSF,4,7/24/48,1,15.0,99,NaN,...,7/15/11,-5.0,3.0,1948-07-24,2011-07-15,62.973306,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}",NaN


In [10]:
print(merged_df['patient_summary'].iloc[0])

{"Subject Demographics": {"Gender": "Male", "Years of Education": 15, "Race": "White", "Age": 68}, "Neuropsychological Battery Summary Scores": {"Montreal Cognitive Assessment": 14, "Modified Benson Figure": 8}}


In [11]:
def add_mri_report(row):
    """Add MRI Report section to mapped_json"""
    # Parse the JSON string to a dictionary
    data = json.loads(row['patient_summary'])
    if pd.notna(row['brain_findings_summary']):
        # Create the MRI Report section
        mri_report = {
            "MRI Report": {
                "MRI features extraction procedure": "A Generalized Additive Model for Location, Scale, and Shape (GAMLSS) was fitted to FastSurfer-derived regional brain volumes acquired from 2,087 healthy controls, following the Brain charts for the human lifespan methodology published in Nature (2022). Data were aggregated from five independent cohorts, and the analysis included region-of-interest (ROI) volumes for the medial and lateral temporal lobes, medial and lateral parietal lobes, frontal, occipital lobes, as well as the inferior lateral ventricles and hippocampus. To ensure a robust normative reference, participants with any known neurodegenerative disease were excluded and subjects with extreme brain volumes (>3.5 absolute z-scores) were excluded during quality control prior to model fitting. The Brain Charts pipeline was implemented to model age- and sex-specific normative centile curves for each ROI. Centile scores (scaled 0-1) were computed for both healthy controls and patient samples. Abnormality was defined based on a one-sided z-score threshold: mild atrophy/enlargement for scores corresponding to 1 standard deviation (z=1), moderate for 1.5 SD (z=1.5), and severe for 2 SD (z=2). For all lobar ROIs, lower centiles indicated greater atrophy, while for ventricular ROIs, higher centiles indicated enlargement (i.e., abnormality). The resulting findings are as follows: ",
                "T1 MRI Findings": str(row['brain_findings_summary'])
            }
        }
        
        if 'Imaging evidence' not in data.keys():
            data['Imaging evidence'] = mri_report
        else:
            # Add the MRI Report to the data
            data['Imaging evidence'].update(mri_report)
        
    # Convert back to JSON string
    return json.dumps(data)

In [12]:
merged_df['patient_summary'] = merged_df.apply(add_mri_report, axis=1)

In [13]:
json.loads(merged_df['patient_summary'].iloc[15])

{'Subject Demographics': {'Gender': 'Male',
  'Years of Education': 18,
  'Race': 'White',
  'Age': 71},
 "Unified Parkinson's Disease Rating Scale (UPDRS)": {"MD Exam - Unified Huntington's Disease Rating Scale": 5,
  "MD Exam - Unified Parkinson's Disease Rating Scale": 2},
 'Geriatric Depression Scale (GDS)': {'Geriatric Depression Scale - Total (30 item)': 0},
 'Activities of Daily Living': {'Schwab and England Activities of Daily Living - Score': 100},
 'Neuropsychological Battery Summary Scores': {'Mini Mental State Exam - Total score': 28,
  'Modified Trails - Completion time, in seconds': 23,
  'Modified Trails - Correct lines': 14,
  'Modified Trails - Errors': 1,
  'Digit Span - Maximum forward recall span': 6,
  'Digit Span - Maximum backward recall span': 4,
  'Verbal Fluency - D Words - Correct D words generated': 16,
  'Verbal Fluency - Animals - Correct animals generated': 26,
  'Boston Naming Test (15-item) - BNT Total': 15,
  'Montreal Cognitive Assessment': 15,
  'Mod

In [14]:
# merged_df.to_csv('/projectnb/vkolagrp/projects/adrd_foundation_model/json_testing_data/nifd_oct_2025.csv', index=False)

In [16]:
merged_df.to_csv(f'{directory_path}/test_gamlss.csv', index=False)